# Phase 5 - Foolproof GUI Model Comparison + OCR

This notebook launches a Gradio app that lets you:
- upload one image
- compare YOLOv8, Faster R-CNN, and RetinaNet detections
- run the **same OCR pipeline** on the best detection from **each** loaded model
- view annotated detections, crops, preprocessing outputs, and a comparison table
- continue working even if one or more models fail to load


In [1]:
# Run once if needed
# %pip install gradio ultralytics torchvision pillow easyocr opencv-python pandas


In [2]:

from pathlib import Path
import json
import re
import warnings
from functools import lru_cache

import numpy as np
import cv2
import pandas as pd
from PIL import Image, ImageDraw
import torch
from torchvision.transforms import functional as F
from torchvision.models.detection import fasterrcnn_resnet50_fpn, retinanet_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.retinanet import RetinaNetClassificationHead
from ultralytics import YOLO
import easyocr
import gradio as gr

warnings.filterwarnings(
    "ignore",
    message="'pin_memory' argument is set as true but no accelerator is found",
)

try:
    PROJECT_ROOT = Path(__file__).resolve().parent
except NameError:
    PROJECT_ROOT = Path().resolve()

CLASS_NAMES = {1: "licence"}
MODEL_COLORS = {
    "YOLOv8": "red",
    "Faster R-CNN": "blue",
    "RetinaNet": "green",
}


def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


DEVICE = get_device()


def find_first_existing(paths):
    for p in paths:
        p = Path(p)
        if p.exists():
            return p
    return None


def recursive_find(filename):
    matches = list(PROJECT_ROOT.rglob(filename))
    return matches[0] if matches else None


def load_json_if_exists(path):
    path = Path(path)
    if path.exists():
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    return None


def locate_yolo_weights():
    candidates = [
        PROJECT_ROOT / "runs" / "plate_detector" / "weights" / "best.pt",
        PROJECT_ROOT / "runs" / "detect" / "plate_detector" / "weights" / "best.pt",
        PROJECT_ROOT / "runs" / "detect" / "train" / "weights" / "best.pt",
        PROJECT_ROOT / "runs" / "detect" / "train2" / "weights" / "best.pt",
    ]
    return find_first_existing(candidates) or recursive_find("best.pt")


def locate_faster_weights():
    candidates = [
        PROJECT_ROOT / "fasterrcnn_outputs" / "best_fasterrcnn_license_plate.pth",
    ]
    return find_first_existing(candidates) or recursive_find("best_fasterrcnn_license_plate.pth")


def locate_retina_weights():
    candidates = [
        PROJECT_ROOT / "retinanet_outputs" / "best_retinanet_license_plate.pth",
    ]
    return find_first_existing(candidates) or recursive_find("best_retinanet_license_plate.pth")


YOLO_WEIGHTS = locate_yolo_weights()
FRCNN_WEIGHTS = locate_faster_weights()
RETINA_WEIGHTS = locate_retina_weights()

FRCNN_RESULTS_JSON = find_first_existing([
    PROJECT_ROOT / "fasterrcnn_outputs" / "fasterrcnn_results.json",
])
RETINA_RESULTS_JSON = find_first_existing([
    PROJECT_ROOT / "retinanet_outputs" / "retinanet_results.json",
])


def build_faster_model(num_classes=2):
    model = fasterrcnn_resnet50_fpn(weights=None, weights_backbone=None)
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    return model


def build_retina_model(num_classes=2):
    model = retinanet_resnet50_fpn(weights=None, weights_backbone=None, min_size=512, max_size=512)
    num_anchors = model.head.classification_head.num_anchors
    in_channels = model.backbone.out_channels
    model.head.classification_head = RetinaNetClassificationHead(
        in_channels=in_channels,
        num_anchors=num_anchors,
        num_classes=num_classes,
    )
    model.score_thresh = 0.5
    model.nms_thresh = 0.5
    model.detections_per_img = 100
    return model


def safe_load_state_dict(model, weights_path):
    state = torch.load(weights_path, map_location=DEVICE)
    if isinstance(state, dict) and "model_state_dict" in state:
        state = state["model_state_dict"]
    model.load_state_dict(state)
    model.to(DEVICE)
    model.eval()
    return model


@lru_cache(maxsize=1)
def load_yolo_model():
    if YOLO_WEIGHTS is None:
        raise FileNotFoundError("YOLO best.pt not found.")
    return YOLO(str(YOLO_WEIGHTS))


@lru_cache(maxsize=1)
def load_faster_model():
    if FRCNN_WEIGHTS is None:
        raise FileNotFoundError("Faster R-CNN weights not found.")
    model = build_faster_model(num_classes=2)
    return safe_load_state_dict(model, FRCNN_WEIGHTS)


@lru_cache(maxsize=1)
def load_retina_model():
    if RETINA_WEIGHTS is None:
        raise FileNotFoundError("RetinaNet weights not found.")
    model = build_retina_model(num_classes=2)
    return safe_load_state_dict(model, RETINA_WEIGHTS)


@lru_cache(maxsize=1)
def load_ocr_reader():
    return easyocr.Reader(["en"], gpu=(DEVICE.type == "cuda"))


def try_load(name, loader):
    try:
        model = loader()
        return model, None
    except Exception as e:
        return None, str(e)


YOLO_MODEL, YOLO_LOAD_ERROR = try_load("YOLO", load_yolo_model)
FRCNN_MODEL, FRCNN_LOAD_ERROR = try_load("FRCNN", load_faster_model)
RETINA_MODEL, RETINA_LOAD_ERROR = try_load("RETINA", load_retina_model)
OCR_READER, OCR_LOAD_ERROR = try_load("OCR", load_ocr_reader)

print(f"Using device: {DEVICE}")
print(f"YOLO weights: {YOLO_WEIGHTS}")
print(f"Faster R-CNN weights: {FRCNN_WEIGHTS}")
print(f"RetinaNet weights: {RETINA_WEIGHTS}")
if YOLO_LOAD_ERROR:
    print("YOLO loading error:", YOLO_LOAD_ERROR)
if FRCNN_LOAD_ERROR:
    print("Faster R-CNN loading error:", FRCNN_LOAD_ERROR)
if RETINA_LOAD_ERROR:
    print("RetinaNet loading error:", RETINA_LOAD_ERROR)
if OCR_LOAD_ERROR:
    print("EasyOCR loading error:", OCR_LOAD_ERROR)


def draw_boxes(pil_image, boxes, scores, labels=None, score_thresh=0.25, color="red"):
    img = pil_image.copy()
    draw = ImageDraw.Draw(img)

    for i, box in enumerate(boxes):
        score = float(scores[i]) if scores is not None and i < len(scores) else 1.0
        if score < score_thresh:
            continue

        x1, y1, x2, y2 = [float(v) for v in box]
        draw.rectangle([x1, y1, x2, y2], outline=color, width=3)

        label_text = ""
        if labels is not None and i < len(labels):
            label_value = labels[i]
            if isinstance(label_value, (int, np.integer)):
                label_text = CLASS_NAMES.get(int(label_value), str(label_value))
            else:
                label_text = str(label_value)

        text = f"{label_text} {score:.2f}".strip()
        draw.text((x1, max(0, y1 - 18)), text, fill=color)

    return img


def pil_to_tensor(image):
    return F.to_tensor(image).to(DEVICE)


def empty_prediction_result(model_name, message, pil_image=None):
    base = pil_image.copy() if pil_image is not None else None
    return {
        "model_name": model_name,
        "annotated": base,
        "detection_count": 0,
        "status": message,
        "best_box": None,
        "best_score": None,
        "crop": None,
        "preprocessed": None,
        "raw_ocr_text": "",
        "cleaned_text": "",
        "ocr_confidence": None,
        "detected": False,
    }


def predict_yolo(pil_image, conf_thresh):
    if YOLO_MODEL is None:
        return empty_prediction_result("YOLOv8", f"Model not loaded: {YOLO_LOAD_ERROR}", pil_image)

    results = YOLO_MODEL.predict(source=np.array(pil_image), conf=conf_thresh, verbose=False)[0]
    if results.boxes is None or len(results.boxes) == 0:
        return empty_prediction_result("YOLOv8", "No detections", pil_image)

    boxes = results.boxes.xyxy.cpu().numpy()
    scores = results.boxes.conf.cpu().numpy()
    labels = ["licence"] * len(boxes)
    annotated = draw_boxes(pil_image, boxes, scores, labels=labels, score_thresh=conf_thresh, color=MODEL_COLORS["YOLOv8"])

    best_idx = int(np.argmax(scores))
    return {
        "model_name": "YOLOv8",
        "annotated": annotated,
        "detection_count": int(len(boxes)),
        "status": f"Detections: {len(boxes)}",
        "best_box": boxes[best_idx],
        "best_score": float(scores[best_idx]),
        "crop": None,
        "preprocessed": None,
        "raw_ocr_text": "",
        "cleaned_text": "",
        "ocr_confidence": None,
        "detected": True,
    }


def predict_torch_detector(model, model_name, pil_image, conf_thresh):
    if model is None:
        err = FRCNN_LOAD_ERROR if model_name == "Faster R-CNN" else RETINA_LOAD_ERROR
        return empty_prediction_result(model_name, f"Model not loaded: {err}", pil_image)

    img_tensor = pil_to_tensor(pil_image)
    with torch.no_grad():
        pred = model([img_tensor])[0]

    boxes = pred["boxes"].detach().cpu().numpy() if "boxes" in pred else np.empty((0, 4))
    scores = pred["scores"].detach().cpu().numpy() if "scores" in pred else np.empty((0,))
    labels = pred["labels"].detach().cpu().numpy() if "labels" in pred else np.empty((0,))

    kept = scores >= conf_thresh
    boxes = boxes[kept]
    scores = scores[kept]
    labels = labels[kept]

    if len(boxes) == 0:
        return empty_prediction_result(model_name, "No detections", pil_image)

    annotated = draw_boxes(pil_image, boxes, scores, labels=labels, score_thresh=conf_thresh, color=MODEL_COLORS[model_name])
    best_idx = int(np.argmax(scores))
    return {
        "model_name": model_name,
        "annotated": annotated,
        "detection_count": int(len(boxes)),
        "status": f"Detections: {len(boxes)}",
        "best_box": boxes[best_idx],
        "best_score": float(scores[best_idx]),
        "crop": None,
        "preprocessed": None,
        "raw_ocr_text": "",
        "cleaned_text": "",
        "ocr_confidence": None,
        "detected": True,
    }


def clean_plate_text(text):
    text = text.upper()
    return re.sub(r"[^A-Z0-9]", "", text)


def expand_box(x1, y1, x2, y2, img_w, img_h, pad_ratio=0.08):
    box_w = max(0, x2 - x1)
    box_h = max(0, y2 - y1)
    pad_x = int(box_w * pad_ratio)
    pad_y = int(box_h * pad_ratio)

    x1 = max(0, int(x1) - pad_x)
    y1 = max(0, int(y1) - pad_y)
    x2 = min(img_w, int(x2) + pad_x)
    y2 = min(img_h, int(y2) + pad_y)
    return x1, y1, x2, y2


def crop_plate_from_box(pil_image, box, pad_ratio=0.08):
    if box is None:
        return None

    image_rgb = np.array(pil_image.convert("RGB"))
    h, w = image_rgb.shape[:2]
    x1, y1, x2, y2 = [int(v) for v in box]
    x1, y1, x2, y2 = expand_box(x1, y1, x2, y2, w, h, pad_ratio=pad_ratio)

    if x2 <= x1 or y2 <= y1:
        return None

    crop_rgb = image_rgb[y1:y2, x1:x2]
    if crop_rgb.size == 0:
        return None

    return Image.fromarray(crop_rgb)


def preprocess_plate_for_ocr(pil_crop):
    if pil_crop is None:
        return None
    crop_rgb = np.array(pil_crop.convert("RGB"))
    crop_bgr = cv2.cvtColor(crop_rgb, cv2.COLOR_RGB2BGR)
    gray = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2GRAY)
    gray = cv2.resize(gray, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)
    gray = cv2.GaussianBlur(gray, (3, 3), 0)
    _, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return Image.fromarray(thresh)


def run_ocr_on_plate(processed_pil):
    if OCR_READER is None:
        return "OCR not loaded", "", None
    if processed_pil is None:
        return "No crop available", "", None

    arr = np.array(processed_pil)
    results = OCR_READER.readtext(arr)
    if not results:
        return "No text detected", "", 0.0

    raw_text = " ".join([item[1] for item in results]).strip()
    cleaned_text = clean_plate_text(raw_text)
    avg_conf = float(sum(float(item[2]) for item in results) / len(results))
    return raw_text, cleaned_text, avg_conf


def apply_shared_ocr_pipeline(pred_result, pil_image, pad_ratio):
    pred_result = dict(pred_result)
    crop = crop_plate_from_box(pil_image, pred_result.get("best_box"), pad_ratio=pad_ratio)
    processed = preprocess_plate_for_ocr(crop)
    raw_text, cleaned_text, ocr_conf = run_ocr_on_plate(processed)

    pred_result["crop"] = crop
    pred_result["preprocessed"] = processed
    pred_result["raw_ocr_text"] = raw_text
    pred_result["cleaned_text"] = cleaned_text
    pred_result["ocr_confidence"] = ocr_conf
    return pred_result


def build_metrics_markdown():
    frcnn = load_json_if_exists(FRCNN_RESULTS_JSON) if FRCNN_RESULTS_JSON else None
    retina = load_json_if_exists(RETINA_RESULTS_JSON) if RETINA_RESULTS_JSON else None

    lines = [
        "### Environment and saved metrics",
        "",
        f"- **Device used by GUI:** `{DEVICE}`",
        f"- **YOLO loaded:** `{YOLO_MODEL is not None}`",
        f"- **Faster R-CNN loaded:** `{FRCNN_MODEL is not None}`",
        f"- **RetinaNet loaded:** `{RETINA_MODEL is not None}`",
        f"- **EasyOCR loaded:** `{OCR_READER is not None}`",
        "",
    ]

    if frcnn:
        tm = frcnn.get("test_metrics", {})
        lines += [
            "**Faster R-CNN**",
            f"- Best val F1: `{frcnn.get('best_val_f1', 'N/A')}`",
            f"- Test precision: `{tm.get('precision', 'N/A')}`",
            f"- Test recall: `{tm.get('recall', 'N/A')}`",
            f"- Test F1: `{tm.get('f1', 'N/A')}`",
            "",
        ]

    if retina:
        tm = retina.get("test_metrics", {})
        lines += [
            "**RetinaNet**",
            f"- Best val F1: `{retina.get('best_val_f1', 'N/A')}`",
            f"- Test precision: `{tm.get('precision', 'N/A')}`",
            f"- Test recall: `{tm.get('recall', 'N/A')}`",
            f"- Test F1: `{tm.get('f1', 'N/A')}`",
            "",
        ]

    if not frcnn and not retina:
        lines.append("No JSON metrics files were found yet. The GUI still works if the model weights exist.")

    if YOLO_LOAD_ERROR:
        lines.append(f"- YOLO load error: `{YOLO_LOAD_ERROR}`")
    if FRCNN_LOAD_ERROR:
        lines.append(f"- Faster R-CNN load error: `{FRCNN_LOAD_ERROR}`")
    if RETINA_LOAD_ERROR:
        lines.append(f"- RetinaNet load error: `{RETINA_LOAD_ERROR}`")
    if OCR_LOAD_ERROR:
        lines.append(f"- EasyOCR load error: `{OCR_LOAD_ERROR}`")

    return "\n".join(lines)


def compare_models_and_ocr(image, conf_thresh, ocr_pad_ratio):
    if image is None:
        raise gr.Error("Please upload an image first.")

    if not isinstance(image, Image.Image):
        image = Image.fromarray(image)
    image = image.convert("RGB")

    outputs = []
    outputs.append(apply_shared_ocr_pipeline(predict_yolo(image, conf_thresh), image, ocr_pad_ratio))
    outputs.append(apply_shared_ocr_pipeline(predict_torch_detector(FRCNN_MODEL, "Faster R-CNN", image, conf_thresh), image, ocr_pad_ratio))
    outputs.append(apply_shared_ocr_pipeline(predict_torch_detector(RETINA_MODEL, "RetinaNet", image, conf_thresh), image, ocr_pad_ratio))

    rows = []
    for result in outputs:
        rows.append({
            "Model": result["model_name"],
            "Detected": "Yes" if result["detected"] else "No",
            "Detections": result["detection_count"],
            "Detector confidence": None if result["best_score"] is None else round(float(result["best_score"]), 4),
            "OCR raw text": result["raw_ocr_text"],
            "OCR cleaned text": result["cleaned_text"],
            "OCR confidence": None if result["ocr_confidence"] is None else round(float(result["ocr_confidence"]), 4),
            "Status": result["status"],
        })

    comparison_df = pd.DataFrame(rows, columns=[
        "Model", "Detected", "Detections", "Detector confidence",
        "OCR raw text", "OCR cleaned text", "OCR confidence", "Status"
    ])

    summary_lines = [
        "### Prediction and OCR summary",
        "",
    ]
    for result in outputs:
        det_conf = "N/A" if result["best_score"] is None else f"{float(result['best_score']):.4f}"
        ocr_conf = "N/A" if result["ocr_confidence"] is None else f"{float(result['ocr_confidence']):.4f}"
        cleaned = result["cleaned_text"] if result["cleaned_text"] else "N/A"
        summary_lines += [
            f"**{result['model_name']}**",
            f"- Status: {result['status']}",
            f"- Best detector confidence: {det_conf}",
            f"- OCR cleaned text: {cleaned}",
            f"- OCR confidence: {ocr_conf}",
            "",
        ]
    summary_md = "\n".join(summary_lines)

    return (
        outputs[0]["annotated"], outputs[1]["annotated"], outputs[2]["annotated"],
        outputs[0]["crop"], outputs[1]["crop"], outputs[2]["crop"],
        outputs[0]["preprocessed"], outputs[1]["preprocessed"], outputs[2]["preprocessed"],
        summary_md, comparison_df
    )


with gr.Blocks(title="License Plate Detector Comparison + OCR") as demo:
    gr.Markdown("# License Plate Detector GUI + OCR")
    gr.Markdown(
        "Upload one image to compare **YOLOv8**, **Faster R-CNN**, and **RetinaNet**. "
        "Each loaded model goes through the same pipeline: **detect → best box → crop → preprocess → EasyOCR**."
    )

    with gr.Row():
        with gr.Column(scale=1):
            input_image = gr.Image(type="pil", label="Upload image")
            conf_slider = gr.Slider(minimum=0.05, maximum=0.95, value=0.25, step=0.05, label="Detection confidence threshold")
            pad_slider = gr.Slider(minimum=0.00, maximum=0.30, value=0.08, step=0.01, label="OCR crop padding ratio")
            run_btn = gr.Button("Run comparison + OCR", variant="primary")
            metrics_md = gr.Markdown(value=build_metrics_markdown())

        with gr.Column(scale=2):
            summary_md = gr.Markdown("### Prediction and OCR summary\nUpload an image, then click **Run comparison + OCR**.")
            result_table = gr.Dataframe(
                headers=["Model", "Detected", "Detections", "Detector confidence", "OCR raw text", "OCR cleaned text", "OCR confidence", "Status"],
                datatype=["str", "str", "number", "number", "str", "str", "number", "str"],
                row_count=3,
                col_count=(8, "fixed"),
                label="Per-model OCR comparison"
            )

    with gr.Row():
        yolo_out = gr.Image(label="YOLOv8 result")
        frcnn_out = gr.Image(label="Faster R-CNN result")
        retina_out = gr.Image(label="RetinaNet result")

    with gr.Row():
        yolo_crop = gr.Image(label="YOLOv8 best crop")
        frcnn_crop = gr.Image(label="Faster R-CNN best crop")
        retina_crop = gr.Image(label="RetinaNet best crop")

    with gr.Row():
        yolo_pre = gr.Image(label="YOLOv8 OCR preprocessing")
        frcnn_pre = gr.Image(label="Faster R-CNN OCR preprocessing")
        retina_pre = gr.Image(label="RetinaNet OCR preprocessing")

    run_btn.click(
        fn=compare_models_and_ocr,
        inputs=[input_image, conf_slider, pad_slider],
        outputs=[
            yolo_out, frcnn_out, retina_out,
            yolo_crop, frcnn_crop, retina_crop,
            yolo_pre, frcnn_pre, retina_pre,
            summary_md, result_table,
        ],
    )

if __name__ == "__main__":
    demo.launch()


Using CPU. Note: This module is much faster with a GPU.


Using device: cpu
YOLO weights: C:\Users\jjrjg\OneDrive\Desktop\CMPS 261 Project\Plate-Detection-Project\runs\detect\runs\plate_detector\weights\best.pt
Faster R-CNN weights: C:\Users\jjrjg\OneDrive\Desktop\CMPS 261 Project\Plate-Detection-Project\fasterrcnn_outputs\best_fasterrcnn_license_plate.pth
RetinaNet weights: C:\Users\jjrjg\OneDrive\Desktop\CMPS 261 Project\Plate-Detection-Project\retinanet_outputs\best_retinanet_license_plate.pth


C:\Users\jjrjg\OneDrive\Desktop\CMPS 261 Project\Plate-Detection-Project\venv\Lib\site-packages\gradio\components\dataframe.py:192: UserWarning: The `col_count` parameter is deprecated and will be removed. Please use `column_count` instead.
  warnings.warn(


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
